# 5주차. 카나가 지난 대화를 불러오다

**부제:** MCP SQLite 대화/일정 DB로 약속을 결정하기 — search_previous_conversations, extract_candidate_slots, load_member_schedules

## 0. 목표

이번 주에는 로컬 MCP server가 SQLite 대화/일정 DB를 외부 tool 서버처럼 제공한다고 가정한다. agent는 DB를 직접 읽지 않고 **MCP SQLite tool만 호출**해 단서를 모은다. 단서는 두 곳에 흩어져 있다.

- **말한 가용성**: 팀원이 이전 대화에서 "언제 된다"고 말한 시간
- **기존 일정**: 팀원이 이미 잡아둔 약속(충돌 확인용)

카나는 이 MCP 단서들을 종합해, 모두가 가능하면서 기존 일정과 겹치지 않는 시간을 결정한다.

성취 기준:

- OpenAI API key를 사용하는 LangChain agent 흐름을 실행할 수 있다.
- agent가 직접 DB를 읽지 않고 MCP SQLite tool contract로 두 근거(말한 가용성, 기존 일정)를 모아 충돌 없는 시간을 고르는 trace를 검증할 수 있다.

## 1. 준비

5주차도 실제 OpenAI API를 호출한다. 실제 MCP server/client 구현은 별도 문제 repo에서 작성하지만, 이 노트북에서는 MCP server가 노출할 tool과 같은 이름/입출력 contract를 LangChain `@tool`로 대역해 agent의 tool 선택과 trace를 검증한다.

In [1]:
from __future__ import annotations

import json
import os
import re
import sys
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain.tools import tool
from langchain_openai import ChatOpenAI


def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / ".env").exists() or (candidate / ".env.example").exists():
            return candidate
    raise RuntimeError("repo root를 찾지 못했습니다. repo 안에서 노트북을 실행하세요.")


REPO_ROOT = find_repo_root(Path.cwd())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
os.chdir(REPO_ROOT)

load_dotenv(REPO_ROOT / ".env", override=True)
OPENAI_MODEL = os.getenv("OPENAI_MODEL", "gpt-4o-mini")

if not os.getenv("OPENAI_API_KEY"):
    raise RuntimeError("repo 루트 .env 파일에 OPENAI_API_KEY를 설정한 뒤 다시 실행하세요.")


def make_model(max_tokens: int = 700) -> ChatOpenAI:
    return ChatOpenAI(model=OPENAI_MODEL, temperature=0, max_completion_tokens=max_tokens)


def show_json(value: Any) -> None:
    print(json.dumps(value, ensure_ascii=False, indent=2, default=str))


def final_text(agent_result: dict[str, Any]) -> str:
    return agent_result["messages"][-1].content


def extract_tool_trace(agent_result: dict[str, Any]) -> list[dict[str, Any]]:
    trace: list[dict[str, Any]] = []
    for message in agent_result.get("messages", []):
        for call in getattr(message, "tool_calls", []) or []:
            trace.append({"event": "tool_call", "tool_name": call.get("name"), "arguments": call.get("args", {})})
        if getattr(message, "type", None) == "tool":
            trace.append({"event": "tool_result", "tool_name": getattr(message, "name", None), "content": message.content})
    return trace


# 아래 두 dict는 실제 MCP server 뒤의 SQLite 저장소를 대역한다(인메모리). agent는 이걸 직접 읽지 않고,
# MCP SQLite tool(아래 셀에서 @tool로 대역)을 통해서만 단서를 얻는다.

# --- MCP SQLite 단서 1: 멤버가 "언제 된다고 말했나"(이전 대화) ---
# 주의: last_message(요약)에는 구체 시간을 두지 않는다. 실제 시간은 원문에만 있어,
#       가능 시간을 알려면 extract_candidate_slots(MCP tool)로 원문을 파싱해야 한다.
previous_conversations = [
    {"conversation_id": "conv-a", "title": "A 일정 공유", "last_message": "A가 다음 주 가능한 회의 시간을 공유함", "members": ["A"]},
    {"conversation_id": "conv-b", "title": "B 일정 공유", "last_message": "B가 다음 주 가능한 회의 시간을 공유함", "members": ["B"]},
    {"conversation_id": "conv-c", "title": "C 일정 공유", "last_message": "C가 다음 주 가능한 회의 시간을 공유함", "members": ["C"]},
]
conversation_messages = {
    "conv-a": [{"role": "assistant", "content": "A는 다음 주 화요일 15:00 가능하고, 목요일 14:00도 괜찮아요."}],
    "conv-b": [{"role": "assistant", "content": "B는 다음 주 화요일 15:00과 목요일 14:00 둘 다 가능해요."}],
    "conv-c": [{"role": "assistant", "content": "C는 다음 주 화요일 15:00 좋고, 목요일 14:00도 가능해요."}],
}
# --- MCP SQLite 단서 2: 멤버가 "이미 무엇에 잡혀있나"(기존 일정). 충돌 확인용 ---
# 함정: 대화상 모두 화요일 15:00 가능이라 말하지만, A는 그 시간에 선약이 있다.
member_schedules = {
    "A": [{"day": "화요일", "start_time": "15:00", "title": "기존 팀 회의"}],
    "B": [],
    "C": [],
}
show_json({"model": OPENAI_MODEL, "mcp_sqlite_tools": ["search_previous_conversations", "extract_candidate_slots", "load_member_schedules"]})

/Users/kit.t/Downloads/kanamate-ai-agent-course/.venv/lib/python3.10/site-packages/langgraph/checkpoint/serde/encrypted.py:5: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


{
  "model": "gpt-4o-mini",
  "mcp_sqlite_tools": [
    "search_previous_conversations",
    "extract_candidate_slots",
    "load_member_schedules"
  ]
}


## 2. 개념

MCP는 DB 접근 권한을 agent 내부 코드에서 분리한다. 실제 구현에서는 MCP server가 SQLite를 읽고, agent는 MCP tool을 호출만 한다. 이 노트북에서는 같은 contract를 LangChain `@tool`로 대역해 agent의 tool 선택과 trace를 검증한다.

이번 시나리오의 핵심은 **MCP SQLite가 주는 단서가 두 곳에 흩어져 있다**는 점이다. "언제 된다고 말했나"(대화)와 "이미 무엇에 잡혀있나"(기존 일정)는 서로 다른 MCP tool로 얻는다. 어느 하나라도 빠뜨리면 답이 틀리므로, agent는 두 tool을 모두 거쳐야 한다.

또 하나: agent에게 "어떤 MCP tool을 어떤 순서로 써라"라고 **경로**를 지정하지 않는다. 대신 "근거를 확인하고 충돌을 피하라"는 **결과 속성**만 요구하고, 도구 선택은 agent에 맡긴다. 도구를 언제 쓸지는 프롬프트가 아니라 *정보를 어디에 두느냐*(요약 vs 원문)로 설계한다.

## 3. 기본 개념 실습

MCP server가 노출할 SQLite tool 세 개를 LangChain `@tool`로 대역한다 — 대화 검색, 가능 시간 후보 추출, 기존 일정 조회. 검색 요약에는 구체 시간을 두지 않아, 가능 시간을 알려면 후보 추출 tool을 반드시 거쳐야 한다.

In [2]:
# 아래 tool들은 실제 MCP server가 노출할 SQLite contract를 LangChain @tool로 대역한다(저장소는 위 인메모리 dict).
DAY_TIME = re.compile(r"([월화수목금토일]요일)\s*([0-2]?\d:[0-5]\d)")


@tool("search_previous_conversations", description="MCP(SQLite): 멤버 이름이나 질의로 관련 이전 대화를 conversation_id와 함께 찾는다. 요약에는 구체 시간이 없으니 실제 가능 시간은 후보 추출 도구로 확인한다.")
def search_previous_conversations(query: str, members: list[str] | None = None) -> str:
    members = members or []
    hits = [
        row for row in previous_conversations
        if not members or any(member in row["members"] for member in members) or query in row["last_message"]
    ]
    return json.dumps({"hits": hits}, ensure_ascii=False)


@tool("extract_candidate_slots", description="MCP(SQLite): 찾은 대화들(conversation_id 목록)에서 각 멤버가 가능하다고 말한 (요일, 시간) 후보를 원문에서 뽑는다.")
def extract_candidate_slots(conversation_ids: list[str]) -> str:
    candidates = []
    for conversation_id in conversation_ids:
        member = conversation_id.replace("conv-", "").upper()
        for message in conversation_messages.get(conversation_id, []):
            for day, start_time in DAY_TIME.findall(message["content"]):
                candidates.append({"member": member, "day": day, "start_time": start_time, "source": message["content"]})
    return json.dumps({"candidates": candidates}, ensure_ascii=False)


@tool("load_member_schedules", description="MCP(SQLite): 멤버들이 이미 잡아둔 기존 일정을 가져온다. 후보 시간을 최종 확정하기 전, 그 시간에 충돌하는 기존 일정이 없는지 반드시 이 도구로 확인한다.")
def load_member_schedules(members: list[str]) -> str:
    return json.dumps({"existing": {m: member_schedules.get(m, []) for m in members}}, ensure_ascii=False)


show_json(json.loads(extract_candidate_slots.invoke({"conversation_ids": ["conv-a", "conv-b", "conv-c"]})))

{
  "candidates": [
    {
      "member": "A",
      "day": "화요일",
      "start_time": "15:00",
      "source": "A는 다음 주 화요일 15:00 가능하고, 목요일 14:00도 괜찮아요."
    },
    {
      "member": "A",
      "day": "목요일",
      "start_time": "14:00",
      "source": "A는 다음 주 화요일 15:00 가능하고, 목요일 14:00도 괜찮아요."
    },
    {
      "member": "B",
      "day": "화요일",
      "start_time": "15:00",
      "source": "B는 다음 주 화요일 15:00과 목요일 14:00 둘 다 가능해요."
    },
    {
      "member": "B",
      "day": "목요일",
      "start_time": "14:00",
      "source": "B는 다음 주 화요일 15:00과 목요일 14:00 둘 다 가능해요."
    },
    {
      "member": "C",
      "day": "화요일",
      "start_time": "15:00",
      "source": "C는 다음 주 화요일 15:00 좋고, 목요일 14:00도 가능해요."
    },
    {
      "member": "C",
      "day": "목요일",
      "start_time": "14:00",
      "source": "C는 다음 주 화요일 15:00 좋고, 목요일 14:00도 가능해요."
    }
  ]
}


## 4. 카나메이트 확장 예제

Agent에게 **MCP SQLite tool만** 제공하고, system_prompt에는 도구 이름이나 호출 순서를 적지 않는다. 대신 확인해야 할 근거(가용성·기존 일정)와 "충돌을 피하라"는 속성만 요구한다. agent는 MCP tool을 스스로 골라 단서를 모은다.

함정: 대화상 팀원 모두 화요일 15:00이 가능하다고 말하지만, A는 그 시간에 이미 다른 일정이 있다. 기존 일정 MCP tool을 호출하지 않으면 잘못된 화요일 15:00을 답하게 된다.

In [3]:
scheduling_agent = create_agent(
    model=make_model(1200),
    tools=[search_previous_conversations, extract_candidate_slots, load_member_schedules],
    system_prompt=(
        "너는 카나메이트 일정 조율 agent다. 팀원들의 회의 시간을 정할 때 추측하지 말고, "
        "(1) 각 팀원이 가능하다고 말한 시간과 (2) 각 팀원이 이미 잡아둔 기존 일정을 모두 근거로 확인한 뒤, "
        "모두가 가능하면서 누구의 기존 일정과도 겹치지 않는 시간을 하나 골라 이유와 함께 답한다. "
        "어떤 도구를 어떤 순서로 쓸지는 스스로 판단한다."
    ),
)

request = "팀원 A/B/C와 다음 주 회의 시간을 잡아줘. 모두 되고 기존 일정과 안 겹치는 시간으로."
result = scheduling_agent.invoke({"messages": [{"role": "user", "content": request}]})
trace = extract_tool_trace(result)
print(final_text(result))
show_json(trace)

팀원 A, B, C의 가능한 회의 시간은 다음과 같습니다:

- **화요일 15:00**
- **목요일 14:00**

하지만, 팀원 A는 화요일 15:00에 이미 기존 팀 회의가 잡혀 있습니다. 따라서, 이 시간은 사용할 수 없습니다.

**결론적으로, 모든 팀원이 가능하고 기존 일정과 겹치지 않는 시간은:**

- **목요일 14:00**

이 시간은 A, B, C 모두 가능하며, 기존 일정과도 겹치지 않으므로 회의 시간으로 적합합니다.
[
  {
    "event": "tool_call",
    "tool_name": "search_previous_conversations",
    "arguments": {
      "query": "회의 시간",
      "members": [
        "A",
        "B",
        "C"
      ]
    }
  },
  {
    "event": "tool_call",
    "tool_name": "load_member_schedules",
    "arguments": {
      "members": [
        "A",
        "B",
        "C"
      ]
    }
  },
  {
    "event": "tool_result",
    "tool_name": "search_previous_conversations",
    "content": "{\"hits\": [{\"conversation_id\": \"conv-a\", \"title\": \"A 일정 공유\", \"last_message\": \"A가 다음 주 가능한 회의 시간을 공유함\", \"members\": [\"A\"]}, {\"conversation_id\": \"conv-b\", \"title\": \"B 일정 공유\", \"last_message\": \"B가 다음 주 가능한 회의 시간을 공유함\", \"members\": [\"B\"]}, {\"conversation_id\

## 5. 확장 예제 실행

trace에서 agent가 MCP SQLite tool로 두 단서(가용성·기존 일정)를 모두 확인했는지, 그리고 충돌을 피해 목요일 14:00을 골랐는지 검증한다.

In [4]:
called_tools = [event["tool_name"] for event in trace if event["event"] == "tool_call"]
assert "search_previous_conversations" in called_tools
assert "extract_candidate_slots" in called_tools       # 가용성: 요약에 시간이 없으니 원문 파싱 필수
assert "load_member_schedules" in called_tools          # 충돌: 기존 일정 확인 필수(생략하면 화 15:00 함정)

answer = final_text(result)
assert "목요일" in answer and "14:00" in answer           # 충돌을 피한 정답
show_json({"called_tools": called_tools, "answer": answer})

{
  "called_tools": [
    "search_previous_conversations",
    "load_member_schedules",
    "extract_candidate_slots",
    "extract_candidate_slots",
    "extract_candidate_slots"
  ],
  "answer": "팀원 A, B, C의 가능한 회의 시간은 다음과 같습니다:\n\n- **화요일 15:00**\n- **목요일 14:00**\n\n하지만, 팀원 A는 화요일 15:00에 이미 기존 팀 회의가 잡혀 있습니다. 따라서, 이 시간은 사용할 수 없습니다.\n\n**결론적으로, 모든 팀원이 가능하고 기존 일정과 겹치지 않는 시간은:**\n\n- **목요일 14:00**\n\n이 시간은 A, B, C 모두 가능하며, 기존 일정과도 겹치지 않으므로 회의 시간으로 적합합니다."
}


## 6. 회고

확인 질문:

1. agent가 SQLite DB를 직접 읽지 않고 MCP tool을 거치게 하는 이유는 무엇인가? (DB 접근 분리)
2. 검색 요약에서 구체 시간을 빼면 agent의 MCP tool 호출이 어떻게 달라지는가? (정보 배치와 도구 필요성)
3. system_prompt에 도구 이름·순서를 적지 않고도 두 단서를 모두 확인하게 만든 것은 무엇인가? (경로 강제 vs 속성 강제)
4. load_member_schedules를 생략하면 어떤 오답이 나오는가? 그 이유는?

작은 응용 과제: 모든 팀원의 가능 시간이 기존 일정과 겹쳐 **공통 시간이 없는** 케이스를 만들고, agent가 시간을 지어내는지 "가능한 시간이 없다"고 답하는지 확인한다.